In [1]:
# imports
import os, unicodedata
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langchain_ollama import ChatOllama

In [2]:
# modelo local no Ollama (API nativa: sem /v1 no final da URL)
MODEL = 'qwen3:4b'
BASE_URL = 'http://localhost:11434'
modelo = ChatOllama(model=MODEL, base_url=BASE_URL,
    temperature=0, num_predict=256, validate_model_on_init=False)

print(f'Usando o modelo {MODEL} em {BASE_URL}')

Usando o modelo qwen3:4b em http://localhost:11434


In [3]:
# Exemplo de criação de envelope com llm:
# 1. Criamos a agente Ana
ana = create_agent(
    model=modelo,
    tools=[],
    system_prompt="Você é Ana. Resuma textos de forma curta."
)

# 2. Função que cria o envelope
def criar_envelope(remetente, destinatario, conteudo):
    print (remetente, destinatario, conteudo)
    print (destinatario)
    return {
        "remetente": remetente,
        "destinatario": destinatario,
        "conteudo": conteudo
    }

# 3. Permissões do sistema
permissoes = {
    "supervisor": ["ana"]
}

# 4. Ferramenta de comunicação
@tool
def enviar_para_ana(conteudo: str) -> str:
    """Envia um texto para Ana resumir."""

    envelope = criar_envelope(
        "supervisor", "ana", conteudo
    )

    # Verificamos a autorização
    if envelope["destinatario"] not in permissoes.get(
        envelope["remetente"], []
    ):
        return "Erro: sem autorização"

    # Delegamos a tarefa à Ana
    resposta = ana.invoke({
        "messages": [
            {
                "role": "user",
                "content": envelope["conteudo"]
            }
        ]
    })

    return str(resposta["messages"][-1].content)

# 5. Criamos o supervisor
supervisor = create_agent(
    model=modelo,
    tools=[enviar_para_ana],
    system_prompt=(
        "Você é um supervisor. "
        "Sempre delegue tarefas de resumo à Ana "
        "usando a ferramenta enviar_para_ana."
    )
)

# 6. Executamos o sistema
resultado = supervisor.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Resuma: Sistemas multiagentes permitem "
                "que diferentes agentes especializados "
                "colaborem na resolução de problemas."
            )
        }
    ]
})